XGBoost Model

In [1]:
AGluon_target="GOALS"

In [107]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

class DeepNN(nn.Module):
    def __init__(self, input_dim):
        super(DeepNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1)  # Output layer for regression
        )

    def forward(self, x):
        return self.model(x)



In [1]:
import pandas as pd
df=pd.read_csv("testML4.csv").iloc[:,1:]
max_t=df['time'].max()
names= df['name'].unique()
time_df=pd.DataFrame()
for i in range(len(names)):
    name=names[i]
    first_filtered= df[df['name'] == name]
    times=[]
    filtered = first_filtered[first_filtered["minutes"] > 0]
    for g in range(len(filtered)):
        times.append(max_t-g)
    times.reverse()
    filtered["time"]=times

    time_df=pd.concat([time_df, filtered], axis=0, ignore_index=True)

time_df.to_csv("ML_training2.csv")


df=time_df.copy()
df=df[df['position'].isin(["FWD", "DEF", "MID"])]
df["opposition_xg"] = df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
df["opposition_xgc"] = df.apply(lambda row: row[23] if row[18] else row[21], axis=1)


train_df = df[[
    "name", "time",  # or "GW" if that's your time column
    "expected_goals", "opposition_xgc",
    "minutes","Threat","rolling_Adjusted_XG_historic"
]].copy()

lags_n=13+1

train_df = train_df.sort_values(["name", "time"])

# make sure the two series are numeric
for col in ["opposition_xgc","expected_goals","Threat"]:
    train_df[col] = pd.to_numeric(train_df[col], errors="coerce")

g = train_df.groupby("name", group_keys=False)

# build lags/leads for k in [-1, 1..15] (skip k=0)
new_cols = []
for k in range(-1, lags_n):
    if k == 0:
        continue
    suffix = f"lead{abs(k)}" if k < 0 else f"lag{k}"
    for col in ["opposition_xgc","Threat","expected_goals"]:
        out_col = f"{col}_{suffix}"
        train_df[out_col] = g[col].shift(k)  # k<0 = lead, k>0 = lag
        new_cols.append(out_col)

# Option A (simple): set any NaNs from shifting to 0
train_df[new_cols] = train_df[new_cols].fillna(0)
print(train_df)

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\3564101294.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df["opposition_xg"] = df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\3564101294.py:24: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df["opposition_xgc"] = df.apply(lambda row: row[23] if row[18] else row[21], axis=1)


                  name  time  expected_goals  opposition_xgc  minutes  Threat  \
31384  Aaron_Cresswell    67            0.00            1.15     90.0     0.0   
31385  Aaron_Cresswell    68            0.00            1.63     90.0     1.0   
31386  Aaron_Cresswell    69            0.00            1.19     90.0    18.0   
31387  Aaron_Cresswell    70            0.00            1.41     90.0     8.0   
31388  Aaron_Cresswell    71            0.00            1.34     71.0     0.0   
...                ...   ...             ...             ...      ...     ...   
36683   Çaglar_Söyüncü   119            0.09            0.78     90.0     5.0   
36684   Çaglar_Söyüncü   120            0.18            1.62     90.0    25.0   
36685   Çaglar_Söyüncü   121            0.05            1.66     90.0     6.0   
36686   Çaglar_Söyüncü   122            0.10            1.59     90.0    17.0   
36687   Çaglar_Söyüncü   123            0.00            1.47     90.0     0.0   

       rolling_Adjusted_XG_

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, regularizers

# ------- Prepare features -------
# Use your dataframe `train_df` that already contains lag columns.
# Example: expected_goals_lag1..lag15, opposition_xgc_lag1..lag15

# Build lag column list (1..15); adjust names if yours differ
lag_cols = (
    #[f"expected_goals_lag{k}" for k in range(1, lags_n)] +
    [f"opposition_xgc_lag{k}" for k in range(1, lags_n)]+
    [f"Threat_lag{k}" for k in range(1, lags_n)]+
    [f"expected_goals_lag{k}" for k in range(1, lags_n)]
)


aux_cols = ["minutes", "rolling_Adjusted_XG_historic", "opposition_xgc"]
target_col = "expected_goals"

train_df=train_df[train_df['time']>10]
max_time=train_df["time"].max()

train_df_2 = train_df[train_df["time"] <= max_time-15]

# Keep only rows that have all needed columns
cols_needed = lag_cols + aux_cols + [target_col]
df_nn = train_df_2.dropna(subset=[target_col]).copy()
for c in lag_cols + aux_cols:
    if c not in df_nn.columns:
        df_nn[c] = 0.0  # or raise, depending on your preference

# X / y


X_lags_train = df_nn[lag_cols].fillna(-1).values.astype(np.float32)
X_aux_train  = df_nn[aux_cols].fillna(-1).values.astype(np.float32)
y_train      = df_nn[target_col].fillna(-1).values.astype(np.float32)

y_cls = (y_train > 0).astype("float32")

print(len(X_lags_train))

# Scale inputs (helps deep nets a lot)
scaler_lags = StandardScaler().fit(X_lags_train)
scaler_aux  = StandardScaler().fit(X_aux_train)

Xl_train_s = scaler_lags.transform(X_lags_train)
Xa_train_s = scaler_aux.transform(X_aux_train)

# ------- Model -------
# ---------- model ----------
lags_in = keras.Input(shape=(len(lag_cols),), name="lags")
aux_in  = keras.Input(shape=(len(aux_cols),), name="aux")

# shared trunk (same as yours)
x = layers.Dense(16, kernel_initializer="he_normal",
                 kernel_regularizer=regularizers.l2(1e-3))(lags_in)
x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x); x = layers.Dropout(0.2)(x)
x = layers.Dense(8, kernel_initializer="he_normal",
                 kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)

a = layers.Dense(8, activation="relu", kernel_initializer="he_normal")(aux_in)
h = layers.Concatenate()([x, a])
h = layers.Dense(16, activation="relu", kernel_initializer="he_normal")(h)
h = layers.Dropout(0.1)(h)
h = layers.Dense(8,  activation="relu", kernel_initializer="he_normal")(h)

# Head A: probability of nonzero (spike)
p = layers.Dense(1, activation="sigmoid", name="p_nonzero")(h)
# Head B: magnitude given nonzero — keep nonnegative
mu = layers.Dense(1, activation="softplus", name="mu_pos")(h)

model = keras.Model(inputs=[lags_in, aux_in], outputs=[p, mu])
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
    loss={
        "p_nonzero": "binary_crossentropy",
        "mu_pos": tf.keras.losses.Huber(delta=0.5),
    },
    metrics={
        "p_nonzero": [tf.keras.metrics.AUC(name="auc"),
                      tf.keras.metrics.BinaryAccuracy(name="acc")],
        "mu_pos":    [tf.keras.metrics.MeanAbsoluteError(name="mae"),
                      tf.keras.metrics.RootMeanSquaredError(name="rmse")],
    },
)

history = model.fit(
    x=[Xl_train_s, Xa_train_s],
    y={"p_nonzero": y_cls, "mu_pos": y_train},
    epochs=200,
    batch_size=64,
    verbose=1,
    shuffle=True,
)



23247
Epoch 1/200
364/364 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.7653 - mu_pos_loss: 0.1058 - mu_pos_mae: 0.3926 - mu_pos_rmse: 0.4987 - p_nonzero_acc: 0.6458 - p_nonzero_auc: 0.6913 - p_nonzero_loss: 0.6282
Epoch 2/200
364/364 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5881 - mu_pos_loss: 0.0199 - mu_pos_mae: 0.1271 - mu_pos_rmse: 0.2158 - p_nonzero_acc: 0.7240 - p_nonzero_auc: 0.7977 - p_nonzero_loss: 0.5437
Epoch 3/200
364/364 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5617 - mu_pos_loss: 0.0181 - mu_pos_mae: 0.1119 - mu_pos_rmse: 0.2062 - p_nonzero_acc: 0.7342 - p_nonzero_auc: 0.8137 - p_nonzero_loss: 0.5252
Epoch 4/200
364/364 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5549 - mu_pos_loss: 0.0181 - mu_pos_mae: 0.1083 - mu_pos_rmse: 0.2090 - p_nonzero_acc: 0.7317 - p_nonzero_auc: 0.8140 - p_nonzero_loss: 0.5233
Epoch 5/200
364/364 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5381 - mu_pos_loss: 0.0168 - mu_pos_mae: 0.1041 - mu_pos_rmse: 0.1985 - p_nonzero_acc: 0.7349 - p_nonzero_a

In [6]:
model.save("DNN_Ass2.keras")  # includes architecture + weights + optimizer state

In [5]:
import numpy as np
from sklearn.metrics import mean_squared_error
# config
names1 = ['Mohamed_Salah','Kai_Havertz','Ollie_Watkins','Antoine_Semenyo','Bryan_Mbeumo',
          'João_Pedro Junqueira de Jesus','Danny_Welbeck','Nicolas_Jackson','Jean-Philippe_Mateta',
          'Dominic_Calvert-Lewin','Diogo_Teixeira da Silva','Erling_Haaland','Alexander_Isak',
          'Chris_Wood','Matheus_Santos Carneiro Da Cunha','Dominic_Solanke',
          'Gabriel_dos Santos Magalhães','William_Saliba','Lucas_Digne','Ezri_Konsa Ngoyo',
          'Lewis_Dunk','Levi_Colwill','Antonee_Robinson','Trent_Alexander-Arnold',
          'Andrew_Robertson','Joško_Gvardiol','Rico_Lewis','Diogo_Dalot Teixeira',
          'Dan_Burn','Pedro_Porro','Rayan_Aït-Nouri','Kai_Havertz','Gabriel_Martinelli Silva',
          'Bukayo_Saka','Martin_Ødegaard','Morgan_Rogers','Antoine_Semenyo','Marcus_Tavernier',
          'Bryan_Mbeumo','Noni_Madueke','Cole_Palmer','Eberechi_Eze','Dwight_McNeil',
          'Diogo_Teixeira da Silva','Luis_Díaz','Mohamed_Salah','Phil_Foden',
          'Bruno_Borges Fernandes','Harvey_Barnes','Anthony_Gordon',
          'Morgan_Gibbs-White','Brennan_Johnson','Dejan_Kulusevski','James_Maddison','Jarrod_Bowen','Cody_Gakpo','Iliman_Ndiaye']

N_LAST = 8  # evaluate on the last 8 rows per player
df2=pd.read_csv("ML_training2.csv").iloc[:,1:]

# choose a time column for sorting
time_col = 'kickoff_time' if 'kickoff_time' in train_df.columns else ('time' if 'time' in train_df.columns else ('GW' if 'GW' in train_df.columns else None))

errors = []
baseline_errors = []

for name in names1:
    # pick rows for this player
    player_df = train_df.loc[df['name'] == name].copy()
    player_df2=df2.loc[df2['name'] == name].copy()

    if player_df.empty:
        print(f"{name}: no rows")
        continue

    # sort chronologically
    if time_col:
        if time_col == 'kickoff_time':
            player_df[time_col] = pd.to_datetime(player_df[time_col], errors='coerce')
        player_df = player_df.sort_values(time_col)

    # need lag+aux columns present
    missing_lag = [c for c in lag_cols if c not in player_df.columns]
    missing_aux = [c for c in aux_cols if c not in player_df.columns]
    if missing_lag or missing_aux or target_col not in player_df.columns:
        print(f"{name}: missing cols (lags:{missing_lag}, aux:{missing_aux}, target:{target_col in player_df.columns})")
        continue

    # take last N rows
    Xl = player_df[lag_cols].tail(N_LAST).fillna(-1).values.astype(np.float32)
    Xa = player_df[aux_cols].tail(N_LAST).fillna(-1).values.astype(np.float32)
    y_true = player_df[[target_col]].tail(N_LAST).values.astype(np.float32).ravel()

    if len(y_true) < N_LAST:
        print(f"{name}: not enough rows ({len(y_true)})")
        continue

    # scale using the SAME scalers from training
    Xl_s = scaler_lags.transform(Xl)
    Xa_s = scaler_aux.transform(Xa)

    # model inference
    outs = model.predict([Xl_s, Xa_s], verbose=0)

    # NEW FORMAT HANDLING:
    # - hurdle model: outputs [p_nonzero, mu_pos]  -> y = p * mu
    # - single-head (with cap inside): single output already capped
    if isinstance(outs, (list, tuple)) and len(outs) == 2:
        p_hat, mu_hat = outs
        y_pred = (p_hat.ravel() )
    else:
        y_pred = outs.ravel()

    # optional baseline like your stat_preds (if both cols exist)
    opp_xgc = player_df2.apply(lambda row: row[23] if row[18] else row[21], axis=1)
    player_df2["opposition_xgc"]=opp_xgc
    
    stat_preds = (player_df2['opposition_xgc'] * player_df2['Rolling_adjusted_XG2']).tail(N_LAST).values
    baseline_mse = mean_squared_error(y_true, stat_preds)
    baseline_errors.append(baseline_mse)


    mse = mean_squared_error(y_true, y_pred)
    errors.append(mse)

    print(f"\n{name}")
    print("Actuals     :", np.round(y_true, 4).tolist())
    print("NN predictions:", np.round(y_pred, 4).tolist(), " | MSE:", round(mse, 6))
    if stat_preds is not None:
        print("Baseline    :", np.round(stat_preds, 4).tolist(), " | MSE:", round(baseline_mse, 6))
    print("Running avg MSE (NN):", round(float(np.mean(errors)), 6),
          "| Running avg MSE (baseline):", round(float(np.mean(baseline_errors)), 6) if baseline_errors else "n/a")

#0.0811


C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = player_df2.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = player_df2.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as


Mohamed_Salah
Actuals     : [0.05000000074505806, 0.7900000214576721, 0.11999999731779099, 0.49000000953674316, 0.25999999046325684, 0.4099999964237213, 0.6800000071525574, 0.2199999988079071]
NN predictions: [0.9524999856948853, 0.9850000143051147, 0.9775000214576721, 0.953499972820282, 0.975600004196167, 0.9648000001907349, 0.9682999849319458, 0.9616000056266785]  | MSE: 0.406938
Baseline    : [0.2496, 0.5673, 0.3968, 0.315, 0.2976, 0.4991, 0.4371, 0.4256]  | MSE: 0.038413
Running avg MSE (NN): 0.406938 | Running avg MSE (baseline): 0.038413

Kai_Havertz
Actuals     : [1.0800000429153442, 0.4000000059604645, 0.20000000298023224, 0.5299999713897705, 0.49000000953674316, 0.0, 0.14000000059604645, 0.0]
NN predictions: [0.9718000292778015, 0.9642999768257141, 0.9703999757766724, 0.983299970626831, 0.9369999766349792, 0.5058000087738037, 0.6333000063896179, 0.5861999988555908]  | MSE: 0.271468
Baseline    : [0.5304, 0.462, 0.483, 0.4592, 0.3248, 0.3567, 0.6525, 0.3752]  | MSE: 0.11862
Ru

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = player_df2.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = player_df2.apply(lambda row: row[23] if row[18] else row[21], axis=1)



Antoine_Semenyo
Actuals     : [0.20999999344348907, 0.8700000047683716, 0.10999999940395355, 0.20999999344348907, 0.6600000262260437, 0.12999999523162842, 0.029999999329447746, 0.029999999329447746]
NN predictions: [0.925599992275238, 0.9168999791145325, 0.9150999784469604, 0.9326000213623047, 0.9448000192642212, 0.9261000156402588, 0.9829999804496765, 0.8458999991416931]  | MSE: 0.496695
Baseline    : [0.3002, 0.261, 0.208, 0.336, 0.2831, 0.2268, 0.34, 0.1512]  | MSE: 0.083339
Running avg MSE (NN): 0.42011 | Running avg MSE (baseline): 0.072069

Bryan_Mbeumo
Actuals     : [0.6600000262260437, 0.6800000071525574, 0.09000000357627869, 0.0, 0.10000000149011612, 0.3799999952316284, 0.5600000023841858, 0.07999999821186066]
NN predictions: [0.9556000232696533, 0.945900022983551, 0.9254999756813049, 0.9426000118255615, 0.9128999710083008, 0.9035000205039978, 0.983299970626831, 0.9690999984741211]  | MSE: 0.456169
Baseline    : [0.4515, 0.1958, 0.3302, 0.348, 0.2134, 0.1932, 0.3102, 0.3192] 

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = player_df2.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = player_df2.apply(lambda row: row[23] if row[18] else row[21], axis=1)


...

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = player_df2.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = player_df2.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as


Harvey_Barnes
Actuals     : [0.029999999329447746, 0.10999999940395355, 0.0, 0.0, 0.3100000023841858, 0.029999999329447746, 0.0, 0.029999999329447746]
NN predictions: [0.513700008392334, 0.8787000179290771, 0.4778999984264374, 0.2037000060081482, 0.49230000376701355, 0.43220001459121704, 0.475600004196167, 0.5228999853134155]  | MSE: 0.219856
Baseline    : [0.3026, 0.256, 0.174, 0.1095, 0.2475, 0.2048, 0.24, 0.2505]  | MSE: 0.034822
Running avg MSE (NN): 0.363872 | Running avg MSE (baseline): 0.091398

Anthony_Gordon
Actuals     : [0.019999999552965164, 0.550000011920929, 0.47999998927116394, 0.05000000074505806, 0.12999999523162842, 0.07000000029802322, 0.0, 0.05999999865889549]
NN predictions: [0.5845999717712402, 0.8788999915122986, 0.6237999796867371, 0.6596999764442444, 0.819599986076355, 0.8033999800682068, 0.8259000182151794, 0.6692000031471252]  | MSE: 0.360735
Baseline    : [0.1599, 0.1284, 0.168, 0.1241, 0.2805, 0.2048, 0.225, 0.2171]  | MSE: 0.052035
Running avg MSE (NN): 0

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = player_df2.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = player_df2.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as


Brennan_Johnson
Actuals     : [0.0, 0.0, 0.0, 0.0, 0.0, 0.05999999865889549, 0.0, 0.0]
NN predictions: [0.9217000007629395, 0.3792000114917755, 0.43650001287460327, 0.5138999819755554, 0.21660000085830688, 0.3312000036239624, 0.7457000017166138, 0.413100004196167]  | MSE: 0.286889
Baseline    : [0.3542, 0.308, 0.2432, 0.2926, 0.3024, 0.2502, 0.1944, 0.2278]  | MSE: 0.072799
Running avg MSE (NN): 0.363897 | Running avg MSE (baseline): 0.088897

Dejan_Kulusevski
Actuals     : [0.33000001311302185, 0.05000000074505806, 0.0, 0.0, 0.07999999821186066, 0.019999999552965164, 0.0, 0.03999999910593033]
NN predictions: [0.8999999761581421, 0.8841000199317932, 0.5174999833106995, 0.41530001163482666, 0.8751999735832214, 0.47279998660087585, 0.792900025844574, 0.4578999876976013]  | MSE: 0.387706
Baseline    : [0.1683, 0.2388, 0.1309, 0.1595, 0.163, 0.077, 0.152, 0.1098]  | MSE: 0.01781
Running avg MSE (NN): 0.364346 | Running avg MSE (baseline): 0.087556

James_Maddison
Actuals     : [0.0, 0.0, 

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = player_df2.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = player_df2.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_40196\1310267798.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as

In [120]:

time_df=pd.read_csv("ML_training2.csv").iloc[:,1:]

pred="Assist"

df=time_df.copy()
df["opposition_xg"] = df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
df["opposition_xgc"] = df.apply(lambda row: row[23] if row[18] else row[21], axis=1)

if(pred=="GOALS"):
    time_cols=["name", "time","expected_goals", "opposition_xgc","minutes", "Own_Attacking_form", "rolling_Adjusted_XG_historic"]
    lagged_cols=["expected_goals", "opposition_xgc"]
    aux_cols = ["minutes", "Own_Attacking_form", "rolling_Adjusted_XG_historic", "opposition_xgc"]
    target_col = "expected_goals"

    features=["opposition_xgc","Own_Attacking_form","XG_slope","rolling_shots",
              "minutes","rolling_Threat","rolling_XG_historic","Rolling_adjusted_XG_form","rolling_Adjusted_XG_historic"]
    features_test=["opposition_xgc","Own_Attacking_form","XG_slope","rolling_shots",
              "minutes","rolling_Threat","rolling_XG_historic","Rolling_adjusted_XG_form","rolling_Adjusted_XG_historic"]
    model_path="DNN_XG.pt"

    model_path_time=model = "DNN_XG_2.keras"

if(pred=="Assist"):
    time_cols=["name", "time","expected_assists", "opposition_xgc","minutes", "Own_Attacking_form", "rolling_Adjusted_XA_historic"]
    lagged_cols=["expected_assists", "opposition_xgc"]
    aux_cols = ["minutes", "Own_Attacking_form", "rolling_Adjusted_XA_historic", "opposition_xgc"]
    target_col = "expected_assists"

    features=["opposition_xgc",
               "Own_Attacking_form","Rolling_creativity", "Rolling_adjusted_XA2","Cluster",
               "rolling_XA_historic","minutes","rolling_key_passes","XA_slope"]
    features_test=["opposition_xgc",
               "Own_Attacking_form","Rolling_creativity", "Rolling_adjusted_XA","Cluster",
               "rolling_XA_historic","minutes","rolling_key_passes","XA_slope"]
    model_path="DNN_XA.pt"

    model_path_time=model = "DNN_XA_2.keras"
    


train_df_time = df[time_cols].copy()

lags_n=10+1

train_df_time = train_df_time.sort_values(["name", "time"])

# make sure the two series are numeric
for col in lagged_cols:
    train_df_time[col] = pd.to_numeric(train_df_time[col], errors="coerce")

g = train_df_time.groupby("name", group_keys=False)

# build lags/leads for k in [-1, 1..15] (skip k=0)
new_cols = []
for k in range(-1, lags_n):
    if k == 0:
        continue
    suffix = f"lead{abs(k)}" if k < 0 else f"lag{k}"
    for col in lagged_cols:
        out_col = f"{col}_{suffix}"
        train_df_time[out_col] = g[col].shift(k)  # k<0 = lead, k>0 = lag
        new_cols.append(out_col)

# Option A (simple): set any NaNs from shifting to 0
train_df_time[new_cols] = train_df_time[new_cols].fillna(0)
lag_cols = (
    [f"{lagged_cols[0]}_lag{k}" for k in range(1, lags_n)] +
    [f"{lagged_cols[1]}_lag{k}" for k in range(1, lags_n)]
)


scaler_lags = StandardScaler().fit(train_df[lag_cols])
scaler_aux  = StandardScaler().fit(train_df[aux_cols])


new_data=pd.read_csv("Player_Prediction_set.csv")
new_data["opposition_xgc"] = new_data["played_XGC"].values

train_df_now = df.copy()
DNN_scaler_data=train_df_now[features]
DNN_scaler = StandardScaler().fit(DNN_scaler_data)


import numpy as np
from sklearn.metrics import mean_squared_error
# config
names1 = new_data["name"].unique()

column_list = ["Name", "pred", "position", "GW","opp_stat" ]
# choose a time column for sorting
time_col = 'kickoff_time' if 'kickoff_time' in train_df.columns else ('time' if 'time' in train_df_time.columns else ('GW' if 'GW' in train_df_time.columns else None))

errors = []
try:
    DNN_model = torch.load(model_path, map_location=torch.device('cpu'))
except:
    input_dim = len(features)  # or features_test if needed
    DNN_model = DeepNN(input_dim)
    DNN_model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
DNN_model.eval()
total_preds=[]
for name in names1:
    print(name)
    # pick rows for this player
    new_player_data=new_data.loc[new_data['name'] == name].copy()
    
    new_player_pred_data=new_player_data[aux_cols]    
    player_df = train_df_time.loc[train_df_time['name'] == name].copy().tail(1)
    if(len(player_df)<1):
        test_data=new_player_data[features_test].copy()
        test_data.columns = DNN_scaler_data.columns
        preds=[]
        for g in range(len(test_data)):
            row=test_data.iloc[[g]]
            row2=new_player_data.iloc[[g]]
            val_series_scaled = DNN_scaler.transform(row)
            X_val_tensor = torch.tensor(val_series_scaled, dtype=torch.float32)
            with torch.no_grad():
                predictions = DNN_model(X_val_tensor).numpy().flatten()
            preds.append(predictions[0])
        print("pred")
        print(preds)
    else:
        
        row=player_df[lag_cols].to_numpy() 

        new_player_pred_data.loc[:, lag_cols] = np.repeat(row, len(new_player_pred_data), axis=0)
    
    

        # take last N rows
        Xl = new_player_pred_data[lag_cols].fillna(0).values.astype(np.float32)
        Xa = new_player_pred_data[aux_cols].fillna(0).values.astype(np.float32)
        y_true = [0]*len(new_player_pred_data)
        # scale using the SAME scalers from training
        Xl_s = scaler_lags.transform(Xl)
        Xa_s = scaler_aux.transform(Xa)

        time_model= tf.keras.models.load_model(model_path_time)
        preds = time_model.predict([Xl_s, Xa_s], verbose=0).ravel()
        print("pred")
        print(preds)
    for j in range(len(preds)):
        pred_list=[]
        pred_list.append(name)
        pred_list.append(preds[j])
        pred_list.append(new_player_data["position"].values[0])
        pred_list.append(new_player_data["GW"].values[j])
        pred_list.append(new_player_data["played_XGC"].values[j])
    
        total_preds.append(pred_list)

pred_all_players=pd.DataFrame(total_preds,columns=column_list)

pred_all_players.to_csv(f"DNN_{pred}2.csv")


C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\1516718319.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df["opposition_xg"] = df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\1516718319.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df["opposition_xgc"] = df.apply(lambda row: row[23] if row[18] else row[21], axis=1)


David_Raya Martin


C:\Users\OleJacobSimensen\OneDrive - twoday\Documents\Python_test\venv\Lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\OleJacobSimensen\OneDrive - twoday\Documents\Python_test\venv\Lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


pred
[0.0101055  0.00921327 0.00433773 0.01167339 0.00764916 0.00511186
 0.01235675 0.01025266]
Kepa_Arrizabalaga


C:\Users\OleJacobSimensen\OneDrive - twoday\Documents\Python_test\venv\Lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\OleJacobSimensen\OneDrive - twoday\Documents\Python_test\venv\Lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


...

pred
[0.01720202 0.03528216 0.03392909 0.02088606 0.04073367 0.0400016
 0.04035697 0.0362895 ]
Tom_Edozie
pred
[0.06317372, 0.06821899, 0.06777628, 0.04543729, 0.0689742, 0.07789588, 0.06336486, 0.0770063]
Enso_González Medina
pred
[0.06317372, 0.06821899, 0.06777628, 0.04543729, 0.0689742, 0.07789588, 0.06336486, 0.0770063]
Joe_Hodge
pred
[0.06317372, 0.06821899, 0.06777628, 0.04543729, 0.0689742, 0.07789588, 0.06336486, 0.0770063]
Jørgen_Strand Larsen


C:\Users\OleJacobSimensen\OneDrive - twoday\Documents\Python_test\venv\Lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\OleJacobSimensen\OneDrive - twoday\Documents\Python_test\venv\Lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


pred
[0.01105434 0.02241209 0.02186849 0.01343061 0.02923657 0.02705923
 0.02512889 0.02930245]
Fábio_Soares Silva
pred
[0.060557842, 0.06429217, 0.06384751, 0.04312798, 0.064904824, 0.07395356, 0.05933743, 0.07278207]
Saša_Kalajdžić
pred
[0.060557842, 0.06429217, 0.06384751, 0.04312798, 0.064904824, 0.07395356, 0.05933743, 0.07278207]
Leon_Chiwome
pred
[0.060557842, 0.06429217, 0.06384751, 0.04312798, 0.064904824, 0.07395356, 0.05933743, 0.07278207]
Nathan_Fraser


C:\Users\OleJacobSimensen\OneDrive - twoday\Documents\Python_test\venv\Lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\OleJacobSimensen\OneDrive - twoday\Documents\Python_test\venv\Lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


pred
[0.01659132 0.03404041 0.03273418 0.02014587 0.03995855 0.03898647
 0.03895989 0.03559745]
Jhon_Arias
pred
[0.06317372, 0.06821899, 0.06777628, 0.04543729, 0.0689742, 0.07789588, 0.06336486, 0.0770063]


In [112]:
print(total_preds)

[['David_Raya Martin', 0.011977855, 'GKP', 1, 1.3199248053158772, 'David_Raya Martin', 0.029580683, 'GKP', 2, 1.9364630534233769, 'David_Raya Martin', 0.0055520027, 'GKP', 3, 0.8579400610859409, 'David_Raya Martin', 0.020929897, 'GKP', 4, 1.6762141982777394, 'David_Raya Martin', 0.0056301, 'GKP', 5, 0.8253740201197124, 'David_Raya Martin', 0.0063738693, 'GKP', 6, 0.9401056096299678, 'David_Raya Martin', 0.017797938, 'GKP', 7, 1.5538235995686611, 'David_Raya Martin', 0.013077669, 'GKP', 8, 1.3780586269712058], ['Kepa_Arrizabalaga', 0.011774881, 'GKP', 1, 1.3199248053158772, 'Kepa_Arrizabalaga', 0.029237885, 'GKP', 2, 1.9364630534233769, 'Kepa_Arrizabalaga', 0.0053868834, 'GKP', 3, 0.8579400610859409, 'Kepa_Arrizabalaga', 0.020686312, 'GKP', 4, 1.6762141982777394, 'Kepa_Arrizabalaga', 0.005623562, 'GKP', 5, 0.8253740201197124, 'Kepa_Arrizabalaga', 0.006263717, 'GKP', 6, 0.9401056096299678, 'Kepa_Arrizabalaga', 0.0175222, 'GKP', 7, 1.5538235995686611, 'Kepa_Arrizabalaga', 0.01285618, 'GKP

In [20]:
import pandas as pd
from sklearn.svm import SVR

#!pip uninstall -y scikit-learn
#!pip install scikit-learn==1.3.2
#!pip install --upgrade u8darts
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from autogluon.timeseries.splitter import ExpandingWindowSplitter
from sklearn.ensemble import RandomForestRegressor
import torch
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

from autogluon.tabular import TabularPredictor
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import random
import numpy as np

from sklearn.decomposition import PCA


df=pd.read_csv("testML4.csv").iloc[:,1:]
max_t=df['time'].max()
names= df['name'].unique()
time_df=pd.DataFrame()
for i in range(len(names)):
    name=names[i]
    first_filtered= df[df['name'] == name]
    times=[]
    filtered = first_filtered[first_filtered["minutes"] > 0]
    for g in range(len(filtered)):
        times.append(max_t-g)
    times.reverse()
    filtered["time"]=times

    time_df=pd.concat([time_df, filtered], axis=0, ignore_index=True)

time_df.to_csv("ML_training2.csv")

if(AGluon_target=="GOALS"):
    features=["opposition_xgc","Own_Attacking_form","XG_slope","rolling_shots",
              "minutes","rolling_Threat","rolling_XG_historic","Rolling_adjusted_XG2","Rolling_adjusted_XG_form","Cluster_XG"]
    features=["opposition_xgc","Own_Attacking_form","XG_slope","rolling_shots",
              "minutes","rolling_Threat","rolling_XG_historic","Rolling_adjusted_XG_form","rolling_Adjusted_XG_historic","Rolling_adjusted_XG2"]
    target="expected_goals"
elif(AGluon_target=="Assist"):
    features=["opposition_xgc",
               "Own_Attacking_form","Rolling_creativity", "Rolling_adjusted_XA2","Cluster",
               "rolling_XA_historic","minutes","rolling_key_passes","XA_slope"]
    target="expected_assists"


df = pd.read_csv("ML_training2.csv").iloc[:, 1:]
names = ['Mohamed_Salah','Kai_Havertz','Ollie_Watkins','Antoine_Semenyo','Bryan_Mbeumo',
         'João_Pedro Junqueira de Jesus','Danny_Welbeck','Nicolas_Jackson','Jean-Philippe_Mateta',
         'Dominic_Calvert-Lewin','Diogo_Teixeira da Silva','Erling_Haaland','Alexander_Isak',
         'Chris_Wood','Matheus_Santos Carneiro Da Cunha','Dominic_Solanke',
         'Gabriel_dos Santos Magalhães','William_Saliba','Lucas_Digne','Ezri_Konsa Ngoyo',
         'Lewis_Dunk','Levi_Colwill0','Antonee_Robinson','Trent_Alexander-Arnold',
         'Andrew_Robertson','Joško_Gvardiol','Rico_Lewis','Diogo_Dalot Teixeira',
         'Dan_Burn','Pedro_Porro','Rayan_Aït-Nouri','Kai_Havertz','Gabriel_Martinelli Silva',
         'Bukayo_Saka','Martin_Ødegaard','Morgan_Rogers','Antoine_Semenyo','Marcus_Tavernier',
         'Bryan_Mbeumo','Noni_Madueke','Cole_Palmer','Eberechi_Eze','Dwight_McNeil',
         'Diogo_Teixeira da Silva','Luis_Díaz','Mohamed_Salah','Phil_Foden',
         'Bruno_Borges Fernandes','Harvey_Barnes','Anthony_Gordon',
         'Morgan_Gibbs-White','Brennan_Johnson','Dejan_Kulusevski','James_Maddison1','Jarrod_Bowen']
errors = []
df=df[df['position'].isin(["FWD", "DEF", "MID"])]

df["opposition_xg"] = df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
df["opposition_xgc"] = df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
# If not forecasting (is_pred == 0), exclude season 30
player_df = df[df["season"] != 30].copy()
player_df=player_df[player_df['time']>10]
#player_df = player_df[player_df["name"].isin(names)]
max_time=player_df["time"].max()

train_df_2 = player_df[player_df["time"] <= max_time-15]  
scaler = StandardScaler()

train_df = train_df_2[features].copy()
train_df=train_df.fillna(0)
train_y=train_df_2[target].copy()
#model_xg = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.05, max_depth=5,min_child_weight=6)
#model_xg = xgb.XGBRegressor(objective='reg:squarederror')
model_xg=SVR(kernel='rbf', C=0.5, epsilon=0.1,gamma=0.1)
model_xg.fit(train_df,train_y)
train_df_scaled = scaler.fit_transform(train_df)

model_xg.fit(train_df_scaled,train_y)
"""
SEED = 42
# Python & NumPy
random.seed(SEED)
np.random.seed(SEED)

# PyTorch
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  # if using multi-GPU

X_train_tensor = torch.tensor(train_df_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(train_y.values, dtype=torch.float32).view(-1, 1)
dataset = TensorDataset(X_train_tensor, y_train_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True)
input_dim = X_train_tensor.shape[1]
model = DeepNN(input_dim)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=0.001)
epochs = 300
model.train()

for epoch in range(epochs):
    epoch_loss = 0
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(loader):.4f}")"""




# Get feature importances


#predictor = TabularPredictor(label=target, problem_type='regression',path=f"Gluon_{AGluon_target}")
#predictor.fit(train_df, presets='medium_quality')


C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\381473947.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df["opposition_xg"] = df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\381473947.py:76: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df["opposition_xgc"] = df.apply(lambda row: row[23] if row[18] else row[21], axis=1)


'\nSEED = 42\n# Python & NumPy\nrandom.seed(SEED)\nnp.random.seed(SEED)\n\n# PyTorch\ntorch.manual_seed(SEED)\ntorch.cuda.manual_seed(SEED)\ntorch.cuda.manual_seed_all(SEED)  # if using multi-GPU\n\nX_train_tensor = torch.tensor(train_df_scaled, dtype=torch.float32)\ny_train_tensor = torch.tensor(train_y.values, dtype=torch.float32).view(-1, 1)\ndataset = TensorDataset(X_train_tensor, y_train_tensor)\nloader = DataLoader(dataset, batch_size=64, shuffle=True)\ninput_dim = X_train_tensor.shape[1]\nmodel = DeepNN(input_dim)\ncriterion = nn.MSELoss()\noptimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=0.001)\nepochs = 300\nmodel.train()\n\nfor epoch in range(epochs):\n    epoch_loss = 0\n    for batch_X, batch_y in loader:\n        optimizer.zero_grad()\n        output = model(batch_X)\n        loss = criterion(output, batch_y)\n        loss.backward()\n        optimizer.step()\n        epoch_loss += loss.item()\n    \n    if (epoch + 1) % 10 == 0:\n        print(f"Epoc

In [21]:

names1=['Mohamed_Salah','Kai_Havertz','Ollie_Watkins','Antoine_Semenyo','Bryan_Mbeumo',
         'João_Pedro Junqueira de Jesus','Danny_Welbeck','Nicolas_Jackson','Jean-Philippe_Mateta',
         'Dominic_Calvert-Lewin','Diogo_Teixeira da Silva','Erling_Haaland','Alexander_Isak',
         'Chris_Wood','Matheus_Santos Carneiro Da Cunha','Dominic_Solanke',
         'Gabriel_dos Santos Magalhães','William_Saliba','Lucas_Digne','Ezri_Konsa Ngoyo',
         'Lewis_Dunk','Levi_Colwill','Antonee_Robinson','Trent_Alexander-Arnold',
         'Andrew_Robertson','Joško_Gvardiol','Rico_Lewis','Diogo_Dalot Teixeira',
         'Dan_Burn','Pedro_Porro','Rayan_Aït-Nouri','Kai_Havertz','Gabriel_Martinelli Silva',
         'Bukayo_Saka','Martin_Ødegaard','Morgan_Rogers','Antoine_Semenyo','Marcus_Tavernier',
         'Bryan_Mbeumo','Noni_Madueke','Cole_Palmer','Eberechi_Eze','Dwight_McNeil',
         'Diogo_Teixeira da Silva','Luis_Díaz','Mohamed_Salah','Phil_Foden',
         'Bruno_Borges Fernandes','Harvey_Barnes','Anthony_Gordon',
         'Morgan_Gibbs-White','Brennan_Johnson','Dejan_Kulusevski','James_Maddison','Jarrod_Bowen']
#model.eval()  # Set model to evaluation mode
errors = []

for i, name in enumerate(names1):
    print(name)
    
    player_df = df[df["season"] != 30].copy()
    player_df.sort_values(by='time', inplace=True)

    test_df = player_df[player_df["name"] == name] 

    test_df["pred"]=test_df["opposition_xgc"]*test_df["Rolling_adjusted_XG2"]

    val_series = test_df[features].copy().iloc[-8:, :]
    

    actuals = test_df[[target]].copy().iloc[-8:, :]
    
    # Skip if there's not enough data
    if val_series.shape[0] < 8:
        print(f"Not enough data for {name}")
        continue

    # Scale features and convert to torch tensor
    val_series_scaled = scaler.transform(val_series)

    
    stat_preds=test_df["pred"].tail(8).values
    print(stat_preds)

    pred_svr=model_xg.predict(val_series_scaled)
    #X_val_tensor = torch.tensor(val_series_scaled, dtype=torch.float32)

    # Predict using the trained DNN
    """ with torch.no_grad():
        predictions = model(X_val_tensor).numpy().flatten()"""

    actuals = actuals.values.flatten()

    mse = mean_squared_error(actuals,stat_preds)
    errors.append(mse)
    print(f"Actuals: {actuals}")
    print(f"Predictions: {stat_preds}")
    print("Running average RMSE:", sum(errors)/len(errors))

#0.0612
#0.060-med scaler


#stat-0.0735

Mohamed_Salah
[0.558  0.6996 0.8774 0.6426 0.38   0.2622 0.4625 0.4563]
Actuals: [0.44 0.09 1.09 0.21 0.09 0.   0.83 0.43]
Predictions: [0.558  0.6996 0.8774 0.6426 0.38   0.2622 0.4625 0.4563]
Running average RMSE: 0.11330930750000001
Kai_Havertz
[0.2461 0.5278 0.4312 0.448  0.432  0.2862 0.3556 0.6356]
Actuals: [0.88 1.08 0.4  0.2  0.53 0.49 0.   0.14]
Predictions: [0.2461 0.5278 0.4312 0.448  0.432  0.2862 0.3556 0.6356]
Running average RMSE: 0.13118219437500003
Ollie_Watkins
[0.474  0.6384 0.3388 0.2581 0.3752 0.364  0.4424 0.3588]


C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\4062588705.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["pred"]=test_df["opposition_xgc"]*test_df["Rolling_adjusted_XG2"]
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\4062588705.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["pred"]=test_df["opposition_xgc"]*test_df["Rolling_adjusted_XG2"]
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\4062588705.py:25: SettingWithCopyWarning: 
A value is

Actuals: [0.   0.28 0.44 0.   0.21 0.36 0.22 0.  ]
Predictions: [0.474  0.6384 0.3388 0.2581 0.3752 0.364  0.4424 0.3588]
Running average RMSE: 0.11393354000000001
Antoine_Semenyo
[0.232  0.207  0.168  0.222  0.1022 0.1988 0.1204 0.2912]
Actuals: [0.19 0.12 0.12 0.18 0.   0.13 0.08 0.46]
Predictions: [0.232  0.207  0.168  0.222  0.1022 0.1988 0.1204 0.2912]
Running average RMSE: 0.08728468250000002
Bryan_Mbeumo
[0.315  0.136  0.2622 0.231  0.3003 0.452  0.2304 0.2877]
Actuals: [0.13 0.   0.6  0.19 0.06 0.18 0.99 0.27]
Predictions: [0.315  0.136  0.2622 0.231  0.3003 0.452  0.2304 0.2877]
Running average RMSE: 0.09176635550000001
João_Pedro Junqueira de Jesus


C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\4062588705.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["pred"]=test_df["opposition_xgc"]*test_df["Rolling_adjusted_XG2"]
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\4062588705.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["pred"]=test_df["opposition_xgc"]*test_df["Rolling_adjusted_XG2"]
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\4062588705.py:25: SettingWithCopyWarning: 
A value is

[0.4872 0.3423 0.299  0.234  0.4186 0.285  0.4824 0.3888]
Actuals: [0.57 0.79 1.12 0.27 0.1  0.   1.72 0.03]
Predictions: [0.4872 0.3423 0.299  0.234  0.4186 0.285  0.4824 0.3888]
Running average RMSE: 0.13325842729166668
Danny_Welbeck
[0.322  0.2166 0.4221 0.3168 0.3339 0.262  0.2622 0.2226]
Actuals: [0.   0.53 0.64 0.22 0.08 0.   0.79 0.66]
Predictions: [0.322  0.2166 0.4221 0.3168 0.3339 0.262  0.2622 0.2226]
Running average RMSE: 0.12960999160714287
Nicolas_Jackson
[0.594  0.4712 0.4379 0.574  0.3375 0.3146 0.2375 0.24  ]
Actuals: [0.04 0.13 0.19 0.39 0.05 0.09 0.12 0.  ]
Predictions: [0.594  0.4712 0.4379 0.574  0.3375 0.3146 0.2375 0.24  ]
Running average RMSE: 0.12470798500000001
Jean-Philippe_Mateta


...

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\4062588705.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["pred"]=test_df["opposition_xgc"]*test_df["Rolling_adjusted_XG2"]
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\4062588705.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["pred"]=test_df["opposition_xgc"]*test_df["Rolling_adjusted_XG2"]
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\4062588705.py:25: SettingWithCopyWarning: 
A value is

Actuals: [0.1  0.   0.16 0.   0.26 0.   0.04 0.  ]
Predictions: [0.2014 0.2945 0.2034 0.2502 0.2664 0.2952 0.4248 0.2096]
Running average RMSE: 0.07838204600543477
Bruno_Borges Fernandes
[0.1696 0.1584 0.1616 0.252  0.2091 0.2576 0.1456 0.2055]
Actuals: [0.13 0.17 0.   0.59 0.05 0.14 0.03 0.25]
Predictions: [0.1696 0.1584 0.1616 0.252  0.2091 0.2576 0.1456 0.2055]
Running average RMSE: 0.07723707550531915
Harvey_Barnes
[0.2538 0.2128 0.2261 0.3708 0.2108 0.2346 0.1404 0.2052]
Actuals: [0.8  0.21 0.05 0.14 0.26 0.37 0.21 0.08]
Predictions: [0.2538 0.2128 0.2261 0.3708 0.2108 0.2346 0.1404 0.2052]
Running average RMSE: 0.07673186333333333
Anthony_Gordon
[0.2538 0.1904 0.2023 0.3502 0.1984 0.2208 0.117  0.1596]
Actuals: [0.   0.06 0.   0.06 0.04 0.   0.   0.02]
Predictions: [0.2538 0.1904 0.2023 0.3502 0.1984 0.2208 0.117  0.1596]
Running average RMSE: 0.07596585665816327
Morgan_Gibbs-White


C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\4062588705.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["pred"]=test_df["opposition_xgc"]*test_df["Rolling_adjusted_XG2"]
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\4062588705.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["pred"]=test_df["opposition_xgc"]*test_df["Rolling_adjusted_XG2"]
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\4062588705.py:25: SettingWithCopyWarning: 
A value is

[0.1298 0.1548 0.1824 0.1344 0.1248 0.2288 0.1644 0.1781]
Actuals: [0.25 0.1  0.33 0.08 0.   0.28 1.14 0.  ]
Predictions: [0.1298 0.1548 0.1824 0.1344 0.1248 0.2288 0.1644 0.1781]
Running average RMSE: 0.07705630865
Brennan_Johnson
[0.3072 0.2392 0.4752 0.3174 0.363  0.1554 0.218  0.284 ]
Actuals: [0.   0.   0.58 0.08 0.   0.   0.   0.48]
Predictions: [0.3072 0.2392 0.4752 0.3174 0.363  0.1554 0.218  0.284 ]
Running average RMSE: 0.07667478406862745
Dejan_Kulusevski
[0.147  0.2211 0.112  0.138  0.165  0.074  0.1323 0.108 ]
Actuals: [0.33 0.05 0.   0.   0.08 0.02 0.   0.04]
Predictions: [0.147  0.2211 0.112  0.138  0.165  0.074  0.1323 0.108 ]
Running average RMSE: 0.07550464519230769
James_Maddison
[0.3819 0.2016 0.2822 0.2176 0.1768 0.3456 0.1932 0.1036]
Actuals: [0.   0.   0.04 0.   0.05 0.   0.07 0.  ]
Predictions: [0.3819 0.2016 0.2822 0.2176 0.1768 0.3456 0.1932 0.1036]
Running average RMSE: 0.0751506178537736
Jarrod_Bowen
[0.2873 0.1258 0.4218 0.2142 0.2754 0.2448 0.3096 0.3996]


C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_26776\4062588705.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["pred"]=test_df["opposition_xgc"]*test_df["Rolling_adjusted_XG2"]


In [27]:
torch.save(model, "DNN_XG.pt")

In [88]:

names1=['Mohamed_Salah','Kai_Havertz1','Ollie_Watkins','Antoine_Semenyo','Bryan_Mbeumo',
         'João_Pedro Junqueira de Jesus','Danny_Welbeck','Nicolas_Jackson','Jean-Philippe_Mateta',
         'Dominic_Calvert-Lewin','Diogo_Teixeira da Silva','Erling_Haaland','Alexander_Isak',
         'Chris_Wood0','Matheus_Santos Carneiro Da Cunha','Dominic_Solanke',
         'Gabriel_dos Santos Magalhães','William_Saliba','Lucas_Digne','Ezri_Konsa Ngoyo',
         'Lewis_Dunk','Levi_Colwill0','Antonee_Robinson','Trent_Alexander-Arnold',
         'Andrew_Robertson','Joško_Gvardiol','Rico_Lewis','Diogo_Dalot Teixeira',
         'Dan_Burn','Pedro_Porro','Rayan_Aït-Nouri','Kai_Havertz0','Gabriel_Martinelli Silva',
         'Bukayo_Saka','Martin_Ødegaard','Morgan_Rogers','Antoine_Semenyo','Marcus_Tavernier',
         'Bryan_Mbeumo','Noni_Madueke','Cole_Palmer0','Eberechi_Eze','Dwight_McNeil',
         'Diogo_Teixeira da Silva','Luis_Díaz','Mohamed_Salah','Phil_Foden',
         'Bruno_Borges Fernandes','Harvey_Barnes0','Anthony_Gordon0',
         'Morgan_Gibbs-White','Brennan_Johnson0','Dejan_Kulusevski','James_Maddison1','Jarrod_Bowen']
errors=[]

for i in range(len(names1)):
    name=names1[i]
    print(name)
    player_df = df[df["season"] != 30].copy()
    player_df.sort_values(by='time', inplace=True)

    test_df = player_df[player_df["name"] == name] 

    val_series=test_df[features].copy().iloc[-8:,:]

    actuals=test_df[[target]].copy().iloc[-8:,:]
    print(actuals)
    val_series_scaled = scaler.transform(val_series) 
    pred=model_xg.predict(val_series_scaled)
    print(pred)
    actuals=actuals.values

    mse = mean_squared_error(actuals, pred)
    errors.append(mse)
    print("Running average RMSE:", sum(errors)/len(errors))


#0.065



Mohamed_Salah
       expected_assists
16412              0.10
16413              0.28
16414              0.06
16415              0.33
16416              0.31
16417              0.39
16418              0.06
16419              0.42
[0.12084979 0.13521362 0.12266825 0.14261299 0.09739843 0.10624736
 0.18756922 0.36440684]
Running average RMSE: 0.025689812235653336
Kai_Havertz1
     expected_assists
292              0.09
293              0.02
294              0.01
295              0.02
296              0.01
297              0.03
298              0.16
299              0.02
[0.12376852 0.0963338  0.07372885 0.09285691 0.07256295 0.09280969
 0.09242227 0.09373068]
Running average RMSE: 0.014982331105311438
Ollie_Watkins
      expected_assists
3048              0.00
3049              0.00
3050              0.07
3051              0.01
3052              0.01
3053              0.02
3054              0.37
3055              0.02
[0.09524304 0.08336907 0.09781004 0.0928314  0.1011106  0.10227799
 0.

TS Mixer

In [48]:
Ts_Mixer_target="GOALS"

In [42]:
#!pip install --upgrade u8darts
from darts.models import TSMixerModel
from darts.dataprocessing.transformers import Scaler
from darts.timeseries import TimeSeries

import pandas as pd

if(Ts_Mixer_target=="GOALS"):
    past_cov=["shots", "Threat"]
    future_cov=["minutes", "Own_Attacking_form", "opposition_xgc","Cluster"]
    input_chunk=8
    hidden_size=20
    hidden_size=16
    random_state=36
    ff_size=16
    ff_size=32
    num_blocks=1
    target="expected_goals"
elif(Ts_Mixer_target=="Assist"):
    past_cov=["key_passes", "creativity"]
    future_cov=["minutes", "Own_Attacking_form", "opposition_xgc","Cluster"]
    input_chunk=10
    hidden_size=20
    random_state=40
    ff_size=32
    num_blocks=1
    target="expected_assists"
    
    
    

# Load your data and define the list of players
df = pd.read_csv("ML_training2.csv").iloc[:, 1:]
names = ['Mohamed_Salah','Kai_Havertz1','Ollie_Watkins','Antoine_Semenyo','Bryan_Mbeumo',
         'João_Pedro Junqueira de Jesus','Danny_Welbeck','Nicolas_Jackson','Jean-Philippe_Mateta',
         'Dominic_Calvert-Lewin','Diogo_Teixeira da Silva','Erling_Haaland','Alexander_Isak',
         'Chris_Wood0','Matheus_Santos Carneiro Da Cunha','Dominic_Solanke',
         'Gabriel_dos Santos Magalhães','William_Saliba','Lucas_Digne','Ezri_Konsa Ngoyo',
         'Lewis_Dunk','Levi_Colwill0','Antonee_Robinson','Trent_Alexander-Arnold',
         'Andrew_Robertson','Joško_Gvardiol','Rico_Lewis','Diogo_Dalot Teixeira',
         'Dan_Burn','Pedro_Porro','Rayan_Aït-Nouri','Kai_Havertz0','Gabriel_Martinelli Silva',
         'Bukayo_Saka','Martin_Ødegaard','Morgan_Rogers','Antoine_Semenyo','Marcus_Tavernier',
         'Bryan_Mbeumo','Noni_Madueke','Cole_Palmer0','Eberechi_Eze','Dwight_McNeil',
         'Diogo_Teixeira da Silva','Luis_Díaz','Mohamed_Salah','Phil_Foden',
         'Bruno_Borges Fernandes','Harvey_Barnes0','Anthony_Gordon0',
         'Morgan_Gibbs-White','Brennan_Johnson0','Dejan_Kulusevski','Jarrod_Bowen']
df = df[df["season"] != 30]
name_counts = df['name'].value_counts()
print(name_counts)
names_more_than_15 = name_counts[name_counts > 30].index.tolist()

name_to_index = {name: idx for idx, name in enumerate(names_more_than_15)}
player_df = df[df["name"].isin(names_more_than_15)]
player_df = player_df[player_df["season"] != 30]
player_df["opposition_xg"] = player_df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
player_df["opposition_xgc"] = player_df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
player_df['expected_goals'] = player_df['expected_goals'].clip(upper=1.5)
player_df.fillna(0, inplace=True)
player_df["name_index"] = player_df["name"].map(name_to_index)

max_time=player_df['time'].max()
print(max_time)

train_df=player_df[player_df["time"]<=max_time-12]

train_target = TimeSeries.from_group_dataframe(train_df, time_col="time", group_cols="name_index",value_cols=target)

train_past_cov = TimeSeries.from_group_dataframe(train_df, time_col="time", group_cols="name_index",value_cols=past_cov)

train_future_cov = TimeSeries.from_group_dataframe(train_df, time_col="time", group_cols="name_index",value_cols=future_cov)
Scaler1=Scaler()

past_cov_scaled=Scaler1.fit_transform(train_past_cov)

# Define model parameters
input_chunk = input_chunk         # number of past time steps used by the model
output_chunk = 8        # forecast horizon
training_length = input_chunk + output_chunk

model=TSMixerModel(
        input_chunk_length=input_chunk, 
        output_chunk_length=output_chunk, 
        hidden_size=hidden_size, 
        ff_size=ff_size,
        num_blocks=num_blocks,
        use_static_covariates=False,
        use_reversible_instance_norm=True,
        activation ="LeakyReLU",
        random_state=random_state)
model.fit(train_target, past_covariates =past_cov_scaled, future_covariates =train_future_cov,epochs=15)

model.save(f"TS_{Ts_Mixer_target}.pkl")
    

ImportError: cannot import name '_check_method_params' from 'sklearn.utils.validation' (C:\Users\OleJacobSimensen\OneDrive - twoday\Documents\Python_test\venv\Lib\site-packages\sklearn\utils\validation.py)

In [ ]:
Test

In [54]:
from sklearn.metrics import mean_squared_error

pred=0

df = pd.read_csv("ML_training2.csv").iloc[:, 1:]
names = ['Mohamed_Salah','Kai_Havertz1','Ollie_Watkins','Antoine_Semenyo','Bryan_Mbeumo',
         'João_Pedro Junqueira de Jesus','Danny_Welbeck','Nicolas_Jackson','Jean-Philippe_Mateta',
         'Dominic_Calvert-Lewin','Diogo_Teixeira da Silva','Erling_Haaland','Alexander_Isak',
         'Chris_Wood0','Matheus_Santos Carneiro Da Cunha','Dominic_Solanke',
         'Gabriel_dos Santos Magalhães','William_Saliba','Lucas_Digne','Ezri_Konsa Ngoyo',
         'Lewis_Dunk','Levi_Colwill0','Antonee_Robinson','Trent_Alexander-Arnold',
         'Andrew_Robertson','Joško_Gvardiol','Rico_Lewis','Diogo_Dalot Teixeira',
         'Dan_Burn','Pedro_Porro','Rayan_Aït-Nouri','Kai_Havertz0','Gabriel_Martinelli Silva',
         'Bukayo_Saka','Martin_Ødegaard','Morgan_Rogers','Antoine_Semenyo','Marcus_Tavernier',
         'Bryan_Mbeumo','Noni_Madueke','Cole_Palmer0','Eberechi_Eze','Dwight_McNeil',
         'Diogo_Teixeira da Silva','Luis_Díaz','Mohamed_Salah','Phil_Foden',
         'Bruno_Borges Fernandes','Harvey_Barnes0','Anthony_Gordon0',
         'Morgan_Gibbs-White','Brennan_Johnson0','Dejan_Kulusevski','Jarrod_Bowen']
player_df=df.copy()
namelist=df["name"].unique()
name_to_index = {name: idx for idx, name in enumerate(namelist)}
player_df["opposition_xg"] = player_df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
player_df["opposition_xgc"] = player_df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
player_df['expected_goals'] = player_df['expected_goals'].clip(upper=1.5)
player_df["name_index"] = player_df["name"].map(name_to_index)

max_ind=player_df["time"].max()
offset=0
print(max_ind)
players_to_use=namelist
if(pred==0):
    print('dddddddddddddddddddddddddddddddddddddd')
    player_df = player_df[player_df["season"] != 30]
    offset=1
    players_to_use=names
errors=[]

pred_df=pd.DataFrame()
for j in range(len(players_to_use)):
    player_preds=[]
    print(players_to_use[j])
    
    new_df=player_df[player_df["name"]==players_to_use[j]]
    
    player_preds.append(players_to_use[j])
    test_df=new_df[new_df["season"]==25]
    if(len(test_df)<1):
        continue

    if(len(new_df)<=(output_chunk+input_chunk)):
        ind_list=list(range(new_df["time"].min()-1, max_ind-output_chunk-11-8*offset, -1))
        dummy_df=pd.DataFrame()
        dummy_df["time"]=ind_list
        dummy_df["name_index"]=new_df["name_index"].values[0]
        new_df=pd.concat([new_df, dummy_df], ignore_index=True)
        new_df.sort_values(by='time', inplace=True)

    new_df.fillna(0, inplace=True)
    test_target = TimeSeries.from_group_dataframe(new_df, time_col="time", group_cols="name_index",value_cols=target)

    test_past_cov = TimeSeries.from_group_dataframe(new_df, time_col="time", group_cols="name_index",value_cols=past_cov)[0][:-output_chunk]

    test_future_cov = TimeSeries.from_group_dataframe(new_df, time_col="time", group_cols="name_index",value_cols=future_cov)[0]


    test_past_cov_scaled=Scaler1.transform(test_past_cov)

    
    loaded_model = TSMixerModel.load(f"TS_{Ts_Mixer_target}.pkl")

    forecast = loaded_model.predict(
                            n=output_chunk,
                            series=test_target[0][:-output_chunk],
                            past_covariates=test_past_cov_scaled,
                            future_covariates=test_future_cov,
    )
    #forecast_inv = scaler_target.inverse_transform(forecast)

    actual = test_target[0][-output_chunk:]
    
    print("Actual:")
    print(actual)
    print("Forecast:")
    print(forecast)
    
    # Convert TimeSeries to numpy arrays using .values()
    mse = mean_squared_error(actual.values(), forecast.values())
    for t in range(len(forecast.values())):
        player_preds.append(forecast.values()[t][0])
    player_preds.append(new_df["position"].values[-1])
    append_df=pd.DataFrame([player_preds], columns=["Name", "p1","p2","p3","p4","p5","p6","p7","p8","position"])
    pred_df = pd.concat([pred_df, append_df], ignore_index=True)

    print(append_df)
    errors.append(mse)
    print(sum(errors)/len(errors))
pred_df.to_csv(f"TS_{Ts_Mixer_target}.csv")
#0.0825
#0.021

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\327595042.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  player_df["opposition_xg"] = player_df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\327595042.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  player_df["opposition_xgc"] = player_df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\327595042.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the cave

114
dddddddddddddddddddddddddddddddddddddd
Mohamed_Salah


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Predicting: |                                                                            | 0/? [00:00<?, ?it/s…

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\327595042.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df.fillna(0, inplace=True)
Only 1 TimeSeries (lists) were provided which is lower than the number of series (n=394) used to fit Scaler. This can result in a mismatch between the series and the underlying transformers.


...

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\327595042.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df.fillna(0, inplace=True)
Only 1 TimeSeries (lists) were provided which is lower than the number of series (n=394) used to fit Scaler. This can result in a mismatch between the series and the underlying transformers.


Actual:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.03]],

       [[0.27]],

       [[0.11]],

       [[0.07]],

       [[0.27]],

       [[0.33]],

       [[0.05]],

       [[0.  ]]])
Coordinates:
  * time       (time) int64 64B 99 100 101 102 103 104 105 106
  * component  (component) <U14 56B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  static_covariates  name_index\ncomponent             ...
    hierarchy:          None
    metadata:           None
Forecast:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.13532088]],

       [[0.1184963 ]],

       [[0.13354191]],

       [[0.17182615]],

       [[0.18249609]],

       [[0.16419797]],

       [[0.20539492]],

       [[0.06296214]]])
Coordinates:
  * time       (time) int64 64B 99 100 101 102 103 104 105 106
  * component  (component) object 8B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    st

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Predicting: |                                                                            | 0/? [00:00<?, ?it/s…

Actual:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.02]],

       [[0.2 ]],

       [[0.  ]],

       [[0.92]],

       [[0.08]],

       [[0.12]],

       [[0.24]],

       [[0.18]]])
Coordinates:
  * time       (time) int64 64B 99 100 101 102 103 104 105 106
  * component  (component) <U14 56B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  static_covariates  name_index\ncomponent             ...
    hierarchy:          None
    metadata:           None
Forecast:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.19200372]],

       [[0.26125916]],

       [[0.25841627]],

       [[0.21423084]],

       [[0.30862872]],

       [[0.28182886]],

       [[0.2631226 ]],

       [[0.2876519 ]]])
Coordinates:
  * time       (time) int64 64B 99 100 101 102 103 104 105 106
  * component  (component) object 8B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    st

In [ ]:
LSTM


In [24]:
LSTM_target="GOALS"

In [43]:
#!pip install --upgrade u8darts
from darts.models import TFTModel
from darts.dataprocessing.transformers import Scaler
from darts.timeseries import TimeSeries
import torch.nn as nn
import torch



import pandas as pd
class RMSELoss(nn.Module):
    def __init__(self):
        super(RMSELoss, self).__init__()
        self.mse = nn.MSELoss()
    def forward(self, yhat, y):
        return torch.sqrt(self.mse(yhat, y))
        
if(LSTM_target=="GOALS"):
    past_cov=["shots", "Threat","Rolling_adjusted_XG2"]
    future_cov=["minutes", "Own_Attacking_form", "opposition_xgc","Cluster"]
    input_chunk=8
    hidden_size=20
    lstm_layers=2
    num_attention_heads=1
    dropout=0.1
    hidden_continuous_size=8
    random_state=36
    target="expected_goals"
elif(LSTM_target=="Assist"):
    past_cov=["key_passes", "creativity"]
    future_cov=["minutes", "Own_Attacking_form", "opposition_xgc","Cluster"]
    input_chunk=8
    hidden_size=64
    lstm_layers=2
    num_attention_heads=2
    dropout=0.1
    hidden_continuous_size=4
    target="expected_assists"
    
    
    

# Load your data and define the list of players
df = pd.read_csv("ML_training2.csv").iloc[:, 1:]
names = ['Mohamed_Salah','Kai_Havertz1','Ollie_Watkins','Antoine_Semenyo','Bryan_Mbeumo',
         'João_Pedro Junqueira de Jesus','Danny_Welbeck','Nicolas_Jackson','Jean-Philippe_Mateta',
         'Dominic_Calvert-Lewin','Diogo_Teixeira da Silva','Erling_Haaland','Alexander_Isak',
         'Chris_Wood0','Matheus_Santos Carneiro Da Cunha','Dominic_Solanke',
         'Gabriel_dos Santos Magalhães','William_Saliba','Lucas_Digne','Ezri_Konsa Ngoyo',
         'Lewis_Dunk','Levi_Colwill0','Antonee_Robinson','Trent_Alexander-Arnold',
         'Andrew_Robertson','Joško_Gvardiol','Rico_Lewis','Diogo_Dalot Teixeira',
         'Dan_Burn','Pedro_Porro','Rayan_Aït-Nouri','Kai_Havertz0','Gabriel_Martinelli Silva',
         'Bukayo_Saka','Martin_Ødegaard','Morgan_Rogers','Antoine_Semenyo','Marcus_Tavernier',
         'Bryan_Mbeumo','Noni_Madueke','Cole_Palmer0','Eberechi_Eze','Dwight_McNeil',
         'Diogo_Teixeira da Silva','Luis_Díaz','Mohamed_Salah','Phil_Foden',
         'Bruno_Borges Fernandes','Harvey_Barnes0','Anthony_Gordon0',
         'Morgan_Gibbs-White','Brennan_Johnson0','Dejan_Kulusevski','Jarrod_Bowen']
df = df[df["season"] != 30]
name_counts = df['name'].value_counts()
print(name_counts)
names_more_than_15 = name_counts[name_counts > 30].index.tolist()

name_to_index = {name: idx for idx, name in enumerate(names_more_than_15)}
player_df = df[df["name"].isin(names_more_than_15)]
player_df = player_df[player_df["season"] != 30]
player_df["opposition_xg"] = player_df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
player_df["opposition_xgc"] = player_df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
player_df['expected_goals'] = player_df['expected_goals'].clip(upper=1.5)
player_df.fillna(0, inplace=True)
player_df["name_index"] = player_df["name"].map(name_to_index)

max_time=player_df['time'].max()
print(max_time)

train_df=player_df[player_df["time"]<=max_time-12]

train_target = TimeSeries.from_group_dataframe(train_df, time_col="time", group_cols="name_index",value_cols=target)

train_past_cov = TimeSeries.from_group_dataframe(train_df, time_col="time", group_cols="name_index",value_cols=past_cov)

train_future_cov = TimeSeries.from_group_dataframe(train_df, time_col="time", group_cols="name_index",value_cols=future_cov)
Scaler1=Scaler()

past_cov_scaled=Scaler1.fit_transform(train_past_cov)

# Define model parameters
input_chunk = input_chunk         # number of past time steps used by the model
output_chunk = 8        # forecast horizon
training_length = input_chunk + output_chunk

model = TFTModel(
    input_chunk_length=input_chunk,
    output_chunk_length=output_chunk,
    hidden_size=hidden_size,
    lstm_layers=lstm_layers,
    num_attention_heads=num_attention_heads,
    dropout=dropout,
    batch_size=32,
    hidden_continuous_size=hidden_continuous_size,
    n_epochs=15,
    force_reset=True,
    use_static_covariates=False,
    loss_fn=RMSELoss()
)
model.fit(train_target, past_covariates =past_cov_scaled, future_covariates =train_future_cov)

model.save(f"LSTM_{LSTM_target}.pkl")
    

ImportError: cannot import name '_check_method_params' from 'sklearn.utils.validation' (C:\Users\OleJacobSimensen\OneDrive - twoday\Documents\Python_test\venv\Lib\site-packages\sklearn\utils\validation.py)

In [47]:
from sklearn.metrics import mean_squared_error

pred=1

df = pd.read_csv("ML_training2.csv").iloc[:, 1:]
names = ['Mohamed_Salah','Kai_Havertz1','Ollie_Watkins','Antoine_Semenyo','Bryan_Mbeumo',
         'João_Pedro Junqueira de Jesus','Danny_Welbeck','Nicolas_Jackson','Jean-Philippe_Mateta',
         'Dominic_Calvert-Lewin','Diogo_Teixeira da Silva','Erling_Haaland','Alexander_Isak',
         'Chris_Wood0','Matheus_Santos Carneiro Da Cunha','Dominic_Solanke',
         'Gabriel_dos Santos Magalhães','William_Saliba','Lucas_Digne','Ezri_Konsa Ngoyo',
         'Lewis_Dunk','Levi_Colwill0','Antonee_Robinson','Trent_Alexander-Arnold',
         'Andrew_Robertson','Joško_Gvardiol','Rico_Lewis','Diogo_Dalot Teixeira',
         'Dan_Burn','Pedro_Porro','Rayan_Aït-Nouri','Kai_Havertz0','Gabriel_Martinelli Silva',
         'Bukayo_Saka','Martin_Ødegaard','Morgan_Rogers','Antoine_Semenyo','Marcus_Tavernier',
         'Bryan_Mbeumo','Noni_Madueke','Cole_Palmer0','Eberechi_Eze','Dwight_McNeil',
         'Diogo_Teixeira da Silva','Luis_Díaz','Mohamed_Salah','Phil_Foden',
         'Bruno_Borges Fernandes','Harvey_Barnes0','Anthony_Gordon0',
         'Morgan_Gibbs-White','Brennan_Johnson0','Dejan_Kulusevski','Jarrod_Bowen']
player_df=df.copy()
namelist=df["name"].unique()
name_to_index = {name: idx for idx, name in enumerate(namelist)}
player_df["opposition_xg"] = player_df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
player_df["opposition_xgc"] = player_df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
player_df['expected_goals'] = player_df['expected_goals'].clip(upper=1.5)
player_df["name_index"] = player_df["name"].map(name_to_index)

max_ind=player_df["time"].max()
offset=0
print(max_ind)
players_to_use=namelist
if(pred==0):
    print('dddddddddddddddddddddddddddddddddddddd')
    player_df = player_df[player_df["season"] != 30]
    offset=1
    players_to_use=names
errors=[]

pred_df=pd.DataFrame()
for j in range(len(players_to_use)):
    player_preds=[]
    print(players_to_use[j])
    
    new_df=player_df[player_df["name"]==players_to_use[j]]
    
    player_preds.append(players_to_use[j])
    test_df=new_df[new_df["season"]==25]
    if(len(test_df)<1):
        continue

    if(len(new_df)<=(output_chunk+input_chunk)):
        ind_list=list(range(new_df["time"].min()-1, max_ind-output_chunk-11-8*offset, -1))
        dummy_df=pd.DataFrame()
        dummy_df["time"]=ind_list
        dummy_df["name_index"]=new_df["name_index"].values[0]
        new_df=pd.concat([new_df, dummy_df], ignore_index=True)
        new_df.sort_values(by='time', inplace=True)

    new_df.fillna(0, inplace=True)
    test_target = TimeSeries.from_group_dataframe(new_df, time_col="time", group_cols="name_index",value_cols=target)

    test_past_cov = TimeSeries.from_group_dataframe(new_df, time_col="time", group_cols="name_index",value_cols=past_cov)[0][:-output_chunk]

    test_future_cov = TimeSeries.from_group_dataframe(new_df, time_col="time", group_cols="name_index",value_cols=future_cov)[0]


    test_past_cov_scaled=Scaler1.transform(test_past_cov)

    
    loaded_model = TFTModel.load(f"LSTM_{LSTM_target}.pkl")

    forecast = loaded_model.predict(
                            n=output_chunk,
                            series=test_target[0][:-output_chunk],
                            past_covariates=test_past_cov_scaled,
                            future_covariates=test_future_cov,
    )
    #forecast_inv = scaler_target.inverse_transform(forecast)

    actual = test_target[0][-output_chunk:]
    
    print("Actual:")
    print(actual)
    print("Forecast:")
    print(forecast)
    
    # Convert TimeSeries to numpy arrays using .values()
    mse = mean_squared_error(actual.values(), forecast.values())
    for t in range(len(forecast.values())):
        player_preds.append(forecast.values()[t][0])
    player_preds.append(new_df["position"].values[-1])
    append_df=pd.DataFrame([player_preds], columns=["Name", "p1","p2","p3","p4","p5","p6","p7","p8","position"])
    pred_df = pd.concat([pred_df, append_df], ignore_index=True)

    print(append_df)
    errors.append(mse)
    print(sum(errors)/len(errors))
pred_df.to_csv(f"TFT_{LSTM_target}.csv")
#0.087

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\1607768472.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  player_df["opposition_xg"] = player_df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\1607768472.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  player_df["opposition_xgc"] = player_df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\1607768472.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the c

114
Fábio_Ferreira Vieira
Gabriel_Fernando de Jesus


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Predicting: |                                                                            | 0/? [00:00<?, ?it/s…

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\1607768472.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df.fillna(0, inplace=True)
Only 1 TimeSeries (lists) were provided which is lower than the number of series (n=394) used to fit Scaler. This can result in a mismatch between the series and the underlying transformers.


...

Only 1 TimeSeries (lists) were provided which is lower than the number of series (n=394) used to fit Scaler. This can result in a mismatch between the series and the underlying transformers.


Actual:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.72]],

       [[0.  ]],

       [[0.04]],

       [[0.02]],

       [[0.03]],

       [[0.04]],

       [[0.02]],

       [[0.03]]])
Coordinates:
  * time       (time) int64 64B 107 108 109 110 111 112 113 114
  * component  (component) <U14 56B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  static_covariates  name_index\ncomponent             ...
    hierarchy:          None
    metadata:           None
Forecast:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.02871703]],

       [[0.01751466]],

       [[0.01740236]],

       [[0.00571614]],

       [[0.0282685 ]],

       [[0.0167055 ]],

       [[0.027473  ]],

       [[0.02700565]]])
Coordinates:
  * time       (time) int64 64B 107 108 109 110 111 112 113 114
  * component  (component) object 8B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Predicting: |                                                                            | 0/? [00:00<?, ?it/s…

Actual:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.6 ]],

       [[0.66]],

       [[0.27]],

       [[0.4 ]],

       [[0.  ]],

       [[0.7 ]],

       [[0.  ]],

       [[0.7 ]]])
Coordinates:
  * time       (time) int64 64B 107 108 109 110 111 112 113 114
  * component  (component) <U14 56B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  static_covariates  name_index\ncomponent             ...
    hierarchy:          None
    metadata:           None
Forecast:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.28536378]],

       [[0.27628441]],

       [[0.25411427]],

       [[0.67543303]],

       [[0.56352412]],

       [[0.31996328]],

       [[0.8341171 ]],

       [[0.29351244]]])
Coordinates:
  * time       (time) int64 64B 107 108 109 110 111 112 113 114
  * component  (component) object 8B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    

In [1]:
future = 5
column_list = []
column_list.append("Name")
for k in range(future):
    column_list.append(f"p{k+1}")
column_list.append("position")
print(column_list)

['Name', 'p1', 'p2', 'p3', 'p4', 'p5', 'position']
